# Rung 45 — R00 / R0 / R1 on the held-out videos

Three arms of `Qwen/Qwen3.6-27B` + NF4, one epoch each, trained on UNAM
(`RESULTS_arm.json` per run). This notebook scores them and reads them against each
other. It runs **rung 42's eval on new arms** — it does not invent one.

| arm | corpus | `lora_dropout` | asks |
|---|---|---|---|
| **R00** | rung 18, 14 415 | 0.0 | it IS the control; re-anchors rung 40 in NF4 |
| **R0** | rung 42 merged, 19 384 | 0.0 | does the 27B gain from more data? |
| **R1** | rung 42 merged, 19 384 | 0.1 | was the over-fit a regularisation deficit? |

## 🔴 Two eval sets, and which arm may see which is not a preference

Rung 42's corpus promoted **30 of the 38 test videos into training**. So:

| arm | test videos in its training set | eval set that is CLEAN for it |
|---|---|---|
| R00 | **0 of 38** | all 6 252 |
| R0 / R1 | **30 of 38** | the **1 283** only |

**Primary for all three: the 1 283** — the only set clean for every arm, so the only one
on which `R0 − R00` is a fair paired comparison. `assert_eval_set_is_legal` RAISES if
anyone points R0 or R1 at the 6 252; that number would be leakage wearing a headline's
clothes.

**The 6 252 is R00-only**, and it is a bridge, never a verdict: R00 shares rung 18's
corpus with rung 40, so `R00 − 40_B_connector_v1` prices NF4 against bf16.

## 🔴 `*_OOD` does not mean the same thing for R00 as for R0/R1 — on the SAME questions

For **R00**, heico is a genuinely unseen procedure. For **R0/R1**, 8 of the 10 heico test
videos are in training, so heico is an unseen **video** of a **seen** procedure. Two videos
carry the whole OOD side.

⇒ an `R0 − R00` win on an `*_OOD` cell is partly **the procedure moving from OOD to ID**,
and is **not** evidence that more data generalises better. The unconfounded read of that
claim is the `*_ID` cells alone. `ood_procedure_in_train` is written per row so the table
carries the distinction.

## Where the numbers come from

Inference is **vLLM**, in **`orena-vllm`**, and both are forced rather than preferred.
Measured on UNAM 2026-08-17: HF materialises the FP8 build at **43.32 GiB** of a 47.37 GiB
card and every generation OOMs (the first smoke died 40/40, caught by G-INFER). vLLM loads
the same checkpoint at the **33.46 GiB** `CLAUDE.md` records, with ~14 GiB spare, at
**0.454 s/question** against HF's 1.13 and an identical 30/50 against gold (rung 44).
The FP8 build itself is produced in `orena-quant` — the only env with `llmcompressor` —
so the two steps use two interpreters on purpose.

Frames are served from `frames_cache`, not decoded from video: UNAM holds **zero `.mp4`**
and every frame this eval needs (6252/6252, 1283/1283, measured 2026-08-17). Same pixels —
the cache stores the raw decord array with no resize — one JPEG q95 round-trip apart. That
cancels on the primary (all three arms read one cache) and does **not** cancel on the bridge.

In [ ]:
# --- bootstrap -------------------------------------------------------------------
import importlib.util, json, logging, os, sys
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent

# 🔴 HF_HOME before the offline flags mean anything, and before any HF import.
# MEASURED on UNAM 2026-08-17: the judge (Qwen/Qwen3-4B) is cached under
# ~/.cache/huggingface. `~/storage/hf_cache` holds the big public weights and NO judge —
# pointing there is what the judge gate two cells below exists to catch, cheaply,
# instead of 40 minutes into an eval. ([[pod-hf-home-is-not-set]])
os.environ.setdefault("HF_HOME", "/home/uaq_user/.cache/huggingface")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

# `_tools` is folder-private, so it is loaded by path rather than installed.
# 🔴 sys.modules registration is REQUIRED, not tidiness: without it @dataclass raises
# `AttributeError: 'NoneType' object has no attribute '__dict__'`, because dataclasses
# resolves the defining module out of sys.modules. Measured 2026-08-17.
_spec = importlib.util.spec_from_file_location(
    "eval_arm45", EXP / "_tools" / "eval_arm45.py")
E = importlib.util.module_from_spec(_spec)
sys.modules["eval_arm45"] = E
_spec.loader.exec_module(E)
print("repo:", REPO, "| exp:", EXP)

In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ------------
SMOKE       = True      # True -> 40 questions on ONE arm. Full: -p SMOKE False
REPO_ROOT   = "/home/uaq_user/storage/repo"
DATA_ROOT   = "/home/uaq_user/storage/orena-data"
FRAMES_CACHE = "/home/uaq_user/storage/frames_cache"
RUNS_ROOT   = "/home/uaq_user/storage/rung45/runs"
# 🔴 The bf16 merge is 52 GB and a card here is 48 GB — it does not fit, and the engine's
# `except Exception` would turn that into 1 283 "Inference Error" strings scored as wrong
# answers, not a crash. FP8 is the established path on this box, not a new idea:
# rung 44 measured 51.75 -> 33.46 GiB in 33.4 s, data-free, and 30/50 exact match against
# bf16's 30/50 (RESULTS_fp8_A.json, RESULTS_fp8_1gpu_real.json).
QUANT_FP8   = True
QUANTIZER   = "/home/uaq_user/storage/repo/experiments/38-gen36-ft-screen/_tools/quantize_fp8.py"
# 🔴 A DIFFERENT interpreter from the one running this notebook: `llmcompressor` lives in
# orena-quant and vllm in orena-vllm, and they cannot share an env (llmcompressor needs
# transformers>=5.9, Unsloth caps at <=5.5 — see context/UNAM_SERVER.md).
QUANTIZER_PYTHON = "/home/uaq_user/storage/envs/orena-quant/bin/python"
ARMS        = ["R00", "R0", "R1"]       # R1 has not run yet; missing arms are skipped
CONTROL_ARM = "R00"
# rung 40's winning arm, bf16, full 6252 — for R00's NF4 bridge only.
BRIDGE_CONTROL_CSV = (
    "/home/uaq_user/storage/repo/experiments/40-gen36-recipe-connector/runs"
    "/40_B_connector_v1/eval/40_B_connector_v1/results.csv")
RUN_BRIDGE  = True      # score R00 on the 6252 as well. NEVER legal for R0/R1.
N_BOOT      = 4000

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) ----
# 🔴 `ensure_paths` FIRST. `eval_arm45` calls it inside its own functions, so importing the
# module does NOT put `src/` or the vendored SDK on sys.path — and `focus` is installed in
# no env on UNAM. Without this line the next import raises ModuleNotFoundError. It needs
# REPO_ROOT, which is why it lives here and not in the bootstrap cell.
E.ensure_paths(REPO_ROOT)
from frame import ledger, metrics

SPLIT_JSON = REPO / "experiments" / "42-merged-corpus" / "RESULTS_split_42.json"

def model_dir(arm: str) -> str:
    # What the eval actually LOADS: the FP8 build when quantizing, else the bf16 merge.
    run = f"45_{arm}_v1"
    return f"{RUNS_ROOT}/{run}/" + ("merged_fp8" if QUANT_FP8 else "merged")

def cfg_for(arm: str, eval_set: str = E.PRIMARY_EVAL) -> "E.EvalConfig":
    run = f"45_{arm}_v1"
    return E.EvalConfig(
        arm=arm, eval_set=eval_set,
        merged_dir=model_dir(arm),
        out_dir=f"{RUNS_ROOT}/{run}/eval" + ("_bridge" if eval_set == E.BRIDGE_EVAL else ""),
        run_name=f"{run}_eval" + ("_bridge" if eval_set == E.BRIDGE_EVAL else ""),
        data_root=DATA_ROOT, repo_root=REPO_ROOT, frames_cache=FRAMES_CACHE,
        split_json=str(SPLIT_JSON), n_boot=N_BOOT, smoke=SMOKE)

# Arms that actually finished. R1 starts when a card frees; a missing arm is SKIPPED,
# never silently treated as absent evidence.
PRESENT = [a for a in ARMS if (Path(RUNS_ROOT) / f"45_{a}_v1" / "merged").is_dir()]
MISSING = [a for a in ARMS if a not in PRESENT]
assert CONTROL_ARM in PRESENT, f"the control {CONTROL_ARM} has no merged checkpoint"
print(f"arms present: {PRESENT}" + (f"   | NOT YET RUN: {MISSING}" if MISSING else ""))

SPLIT = E.held_out_split(cfg_for(CONTROL_ARM))
print(f"held-out: {len(SPLIT['held_videos'])} videos / {SPLIT['n_held']} questions "
      f"({SPLIT['n_heico_videos']} heico videos carry the whole OOD side)")

In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ------
# run_baseline loads the judge only AFTER the whole inference pass, so a missing cache
# fails ~40 minutes in with everything already paid for. RAISES (RULES §7).
from transformers import AutoTokenizer
from frame.config import BaselineConfig

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. Fix the env — do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")

In [ ]:
# --- GATES BEFORE THE GPU (all RAISE) ---------------------------------------------
# 1. Every frame this eval will ask for is already cached. Seconds, vs a FileNotFoundError
#    raised mid-eval with the 27B loaded.
E.assert_cache_covers(cfg_for(CONTROL_ARM), SPLIT["held_items"])
if RUN_BRIDGE:
    E.assert_cache_covers(cfg_for(CONTROL_ARM), SPLIT["all_items"])
print(f"cache gate OK: {SPLIT['n_held']} primary"
      + (f" + {len(SPLIT['all_items'])} bridge" if RUN_BRIDGE else "") + " frames present")

# 2. The training run each arm claims. 🔴 rc=0 is NOT evidence that a run moved a weight:
#    AdamW's decoupled weight decay moves every tensor at zero gradient, so a checkpoint
#    diff cannot separate a real run from a no-op. ([[rc-zero-is-not-evidence]])
for arm in PRESENT:
    j = json.loads((Path(RUNS_ROOT) / f"45_{arm}_v1" / "RESULTS_arm.json").read_text())
    assert j["verdict"] == "OK", f"{arm}: arm verdict is {j['verdict']!r}, not OK"
    print(f"  {arm}: loss {j['train_loss']:.4f} | {j['train_secs']/3600:.1f} h | "
          f"peak {j['peak_vram_gib']:.1f} GiB | corpus {j['dataset']['n_rows']} rows "
          f"| sha {j['dataset']['sha256_rehosted'][:12]}")

# 3. The bridge control must be on disk BEFORE R00 runs, not discovered afterwards.
if RUN_BRIDGE:
    assert Path(BRIDGE_CONTROL_CSV).exists(), f"no bridge control at {BRIDGE_CONTROL_CSV}"
    _b = pd.read_csv(BRIDGE_CONTROL_CSV)
    assert len(_b) == 6252 and _b.qID.nunique() == 6252, f"bridge control is {len(_b)} rows"
    print(f"  bridge control OK: {len(_b)} rows, {_b.video.nunique()} videos")

In [ ]:
# --- FP8: make each arm fit ONE card ----------------------------------------------
# 🔴 Not an optimisation — a fit gate. The bf16 merge is 52 GB against a 48 GB card, and
# `GenericVLMEngine.predict_samples` wraps inference in `except Exception`, so an OOM
# comes back as 1 283 "Inference Error" strings that score as WRONG ANSWERS. The eval
# would report a bad model, not a broken run.
#
# This is the established path on this box, measured by rung 44 and not re-derived here:
#   51.75 GiB -> 33.46 GiB (ratio 0.647) in 33.4 s  (RESULTS_fp8_A.json)
#   FP8 on ONE gpu: 30/50 exact match == bf16's 30/50  (RESULTS_fp8_1gpu_real.json)
# `FP8_DYNAMIC` is DATA-FREE, so this step never touches challenge data.
#
# ⚠️ It DOES add to the bridge's confound stack (NF4 training + JPEG frames + FP8
# inference) against rung 40's bf16 control. Bounded by the 30/50 == 30/50 above, and it
# cancels entirely on the primary: all three arms are quantized identically.
import subprocess, time

if QUANT_FP8:
    for arm in PRESENT:
        src = f"{RUNS_ROOT}/45_{arm}_v1/merged"
        dst = model_dir(arm)
        if (Path(dst) / "config.json").exists():
            print(f"  {arm}: FP8 already present -> {dst}")
            continue
        t0 = time.perf_counter()
        # cwd = the run dir: the quantizer writes RESULTS_fp8_llmcompressor.json to cwd,
        # and that artifact belongs to the run that produced it, not to this experiment
        # folder (storage rule) — otherwise each arm overwrites the last one's.
        Path(f"{RUNS_ROOT}/45_{arm}_v1").mkdir(parents=True, exist_ok=True)
        p = subprocess.run([QUANTIZER_PYTHON, QUANTIZER, src, dst],
                           capture_output=True, text=True, cwd=f"{RUNS_ROOT}/45_{arm}_v1")
        # 🔴 rc is NOT the signal. quantize_fp8.py wraps everything in `except Exception`
        # and never sys.exits, so it returns 0 on failure too. The artifact is the signal;
        # rc is only printed. (Its own step 4 is a generate-check that needs a `frames/`
        # dir of PNGs we do not pass, so a FAIL verdict there is EXPECTED and harmless —
        # the FP8 is already written by step 3. The gate below is what decides.)
        print(f"  {arm}: quantizer rc={p.returncode} in {time.perf_counter()-t0:.0f}s")
        if not (Path(dst) / "config.json").exists():
            print(p.stdout[-3000:]); print(p.stderr[-2000:])
            raise RuntimeError(f"FP8 quantization produced no checkpoint for {arm}")

    # The fit gate. 47.37 GiB usable on this card; a near-miss is what produces the
    # silent-wrong-answer mode above.
    # 🔴 BOTH bounds. An upper bound alone is not a gate: a MISSING directory makes rglob
    # yield nothing, sums to 0.0 GiB, and sails through `< 40`. Rung 44 measured the real
    # figure at 33.46 GiB, so anything far from that band is a broken build, not a small one.
    for arm in PRESENT:
        d = Path(model_dir(arm))
        assert (d / "config.json").exists(), f"{arm}: no config.json at {d}"
        gib = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 2**30
        assert 25 < gib < 40, (
            f"{arm}: build is {gib:.2f} GiB, outside the 25-40 GiB band "
            f"(rung 44 measured 33.46). Too big will OOM into wrong answers; too small "
            f"means the write was truncated.")
        print(f"  {arm}: {gib:.2f} GiB  OK")

In [ ]:
# --- the SWEEP: score each present arm on the 1 283 --------------------------------
# One arm at a time. The merged 27B checkpoints are ~53 GB each and already on disk, so
# nothing is merged or reclaimed here — unlike rung 42, whose sweep merged per epoch.
gold = ledger.gold_from_frame_parquets(Path(DATA_ROOT))
ARM_RES = {}

for arm in (PRESENT[:1] if SMOKE else PRESENT):
    cfg = cfg_for(arm)
    print(f"\n=== {arm} on {cfg.eval_set} ===")
    report = E.score(cfg, SPLIT)
    res = pd.read_csv(E.arm_results_csv(cfg))
    if not SMOKE:
        assert len(res) == SPLIT["n_held"], f"{arm}: {len(res)} of {SPLIT['n_held']} scored"
        metrics.assert_no_dup_qid(res)
        metrics.assert_ood_from_qid(res)
        metrics.assert_all_rows_grouped(res)
    ARM_RES[arm] = {"cfg": cfg, "report": report, "res": res}
    print(f"  bucket_mean={report.get('bucket_mean')}  proxy={report.get('proxy_leaderboard')}")

In [ ]:
# --- 🎯 the PAIRED, VIDEO-CLUSTERED CI vs R00 -------------------------------------
# Effective n is 8 VIDEOS, not 1 283 questions (RULES §13). An unclustered CI would be
# ~10x too narrow and would manufacture significance.
# 🔴 `ALL` is never pooled across ID/OOD — rung 42's shape. The OOD side is 2 videos:
# rung 42 got ALL_ID +0.0705 [+0.0206, +0.1246] while EVERY *_OOD cell included zero.
# An OOD cell failing to exclude zero here is the DESIGN, not a finding.
# Only *_ID may GRANT; *_OOD may only VETO (RULES §S8).
CI = {}
if not SMOKE:
    ctrl = ARM_RES[CONTROL_ARM]["res"]
    for arm in [a for a in ARM_RES if a != CONTROL_ARM]:
        ci = E.paired_cells(ARM_RES[arm]["res"], ctrl, n_boot=N_BOOT, seed=42)
        v = E.verdict(ci)
        CI[arm] = {"ci": ci, "verdict": v}
        print(f"\n=== {arm} MINUS {CONTROL_ARM}, on the {SPLIT['n_held']} ===")
        print(ci.to_string(index=False))
        print(f"  -> {v['verdict']}   won={v['cells_won']}   vetoed={v['cells_vetoed']}")
        print("  ⚠️  *_OOD rests on 2 videos and is confounded: for this arm the heico "
              "PROCEDURE is in training, for the control it is not.")

In [ ]:
# --- class-balanced F1 on `fo_class` — MANDATORY before publishing (RULES §9b) -----
# `fo_class` accuracy is exact SET equality, so it is dominated by the head of a
# long-tailed class distribution and cannot see a tail collapse. `fo_class` is 71 % of
# `object_recognition`, which is where the 27B's whole deficit already sits — it is the
# declared veto cell of this rung, so its tail is the thing to look at.
F1 = {}
if not SMOKE:
    rows = []
    for arm, d in ARM_RES.items():
        preds = metrics.predictions_frame(Path(d["cfg"].out_dir) / d["cfg"].run_name)
        F1[arm] = metrics.class_f1_report(preds, gold, results_df=d["res"], n_boot=2000)
        for cell in ("pooled", "ID", "OOD"):
            b = F1[arm][cell]
            rows.append({"arm": arm, "cell": cell, "n": b["n"], "illegal": b["n_illegal"],
                         "macro_f1": round(b["macro_f1"], 4),
                         "exact_set_acc": round(b["exact_set_acc"], 4),
                         "ci": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]"})
    f1_df = pd.DataFrame(rows)
    print(f1_df.to_string(index=False))

    _per = pd.DataFrame(F1[CONTROL_ARM]["ID"]["per_class"]).T
    print(f"\n--- per class, ID, {CONTROL_ARM} (the tail is the point) ---")
    print("(no fo_class x ID rows)" if _per.empty else
          _per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())
    # 🔴 an illegal class token does not score 0 — FOType.from_name() RAISES (RULES §8b)
    _ill = f1_df[f1_df.illegal > 0]
    if len(_ill):
        print("\n🔴 ILLEGAL fo_class tokens emitted (RULES §8b):")
        print(_ill.to_string(index=False))

In [ ]:
# --- the BRIDGE: R00 on the 6 252, against rung 40's bf16 arm ----------------------
# 🔴 R00 ONLY. `assert_eval_set_is_legal` RAISES for R0/R1 — their corpus contains 30 of
# these 38 videos. This prices NF4 against bf16 on the same corpus and the same recipe.
# ⚠️ It also carries a JPEG round-trip rung 40 did not have (frames_cache vs decord), so
# it is a ceiling on how finely the number can be read — it is a bridge, never a verdict.
BRIDGE = {}
if RUN_BRIDGE and not SMOKE:
    cfg_b = cfg_for(CONTROL_ARM, E.BRIDGE_EVAL)
    rep_b = E.score(cfg_b, SPLIT)
    res_b = pd.read_csv(E.arm_results_csv(cfg_b))
    assert len(res_b) == 6252, f"bridge scored {len(res_b)} of 6252"
    ci_b = E.paired_cells(res_b, pd.read_csv(BRIDGE_CONTROL_CSV), n_boot=N_BOOT, seed=42)
    BRIDGE = {"report": rep_b, "res": res_b, "ci": ci_b}
    print(f"{CONTROL_ARM} (NF4) vs {E.BRIDGE_CONTROL['run']} (bf16), full 6252:")
    for k in ("proxy_leaderboard", "bucket_mean", "aggregation_ID", "object_recognition_ID"):
        print(f"  {k:24s} {rep_b.get(k):.4f}  vs  {E.BRIDGE_CONTROL[k]:.4f}  "
              f"(d = {rep_b.get(k) - E.BRIDGE_CONTROL[k]:+.4f})")
    print(ci_b.to_string(index=False))

In [ ]:
# --- persist: RESULTS.csv + the ledger (full runs only) ----------------------------
if not SMOKE:
    out_rows = []
    for arm, d in ARM_RES.items():
        r = d["report"]
        j = json.loads((Path(RUNS_ROOT) / f"45_{arm}_v1" / "RESULTS_arm.json").read_text())
        row = {
            "run": f"45_{arm}_v1", "arm": arm, "epoch": 1,
            "eval_set": E.PRIMARY_EVAL, "n_eval": SPLIT["n_held"],
            "n_videos": len(SPLIT["held_videos"]), "n_heico_videos": SPLIT["n_heico_videos"],
            # 🔴 the honest label, and it DIFFERS BY ARM on the same questions: R0/R1
            # trained on 8 of the 10 heico test videos, R00 on none.
            "ood_procedure_in_train": arm in E.CONTAMINATED_ARMS,
            "corpus_rows": j["dataset"]["n_rows"],
            "lora_dropout": j["config"]["lora_dropout"],
            "precision": "NF4", "train_loss": j["train_loss"],
            "baseline_run": f"45_{CONTROL_ARM}_v1" if arm != CONTROL_ARM else "",
            **{k: r.get(k) for k in ("bucket_mean", "acc_ID", "acc_OOD", "margin_ID",
                                     "margin_OOD", "aggregation_ID", "object_recognition_ID",
                                     "aggregation_OOD", "object_recognition_OOD",
                                     "proxy_leaderboard")},
            "macro_f1_ID": F1[arm]["ID"]["macro_f1"], "macro_f1_OOD": F1[arm]["OOD"]["macro_f1"],
            "d_bucket_mean_vs_control": (
                r["bucket_mean"] - ARM_RES[CONTROL_ARM]["report"]["bucket_mean"]),
            "verdict": CI.get(arm, {}).get("verdict", {}).get("verdict", "control"),
        }
        metrics.assert_class_f1_reported(row, results_df=d["res"])   # RULES §9b — RAISES
        out_rows.append(row)

    df = pd.DataFrame(out_rows)
    out = EXP / "RESULTS.csv"
    if out.exists():
        df = pd.concat([pd.read_csv(out), df], ignore_index=True)
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / "RESULTS_class_f1.csv", index=False)
    if CI:
        pd.concat([v["ci"].assign(arm=a) for a, v in CI.items()]).to_csv(
            EXP / "RESULTS_paired_ci.csv", index=False)
    if BRIDGE:
        BRIDGE["ci"].to_csv(EXP / "RESULTS_bridge_ci.csv", index=False)

    (EXP / "RESULTS_heldout_eval.json").write_text(json.dumps({
        "eval_set": {"videos": sorted("/".join(v) for v in SPLIT["held_videos"]),
                     "n_questions": SPLIT["n_held"]},
        "arms_scored": list(ARM_RES), "arms_not_run": MISSING,
        "ood_caveat": (
            "On these SAME questions `*_OOD` means unseen PROCEDURE for R00 and unseen "
            "VIDEO of a SEEN procedure for R0/R1 — rung 42's merge promoted 8 of the 10 "
            "heico test videos into their training set. An R0/R1 win on an *_OOD cell is "
            "partly that shift and is NOT evidence that more data generalises better. "
            "Two videos carry the whole OOD side; no *_OOD CI here is readable."),
        "frame_source": (
            "frames_cache (UNAM holds no video). Same frames, one JPEG q95 round-trip "
            "apart; cancels across arms, does NOT cancel on the bridge."),
        "selection_axis": "bucket_mean on the held-out videos (RULES §4c weighting)",
        "granting_cells": ["ALL_ID"], "n_boot": N_BOOT,
    }, indent=2), encoding="utf-8")
    print(df.to_string(index=False))

In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) ---------
if not SMOKE:
    for arm, d in ARM_RES.items():
        preds = metrics.predictions_frame(Path(d["cfg"].out_dir) / d["cfg"].run_name)
        fo = d["res"][d["res"].answer_format == "fo_class"].merge(preds, on="qID")
        print(f"\n--- {arm}: fo_class misses (identity, not cardinality, is the usual one) ---")
        print(fo[fo.correctness == 0].head(8)[["qID", "prediction"]].to_string(index=False))

    g = gold[gold.qID.isin(SPLIT["held_qids"])].copy()
    g["template"] = g["question"].map(metrics.template_of)
    clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
    look = clips.merge(
        metrics.predictions_frame(
            Path(ARM_RES[CONTROL_ARM]["cfg"].out_dir) / ARM_RES[CONTROL_ARM]["cfg"].run_name),
        on="qID")
    if len(look):
        look["gold_n"] = look["answer"].map(metrics.read_count)
        look["pred_n"] = look["prediction"].map(metrics.read_count)
        print(f"\n--- predicted-vs-gold crosstab (Clips), {CONTROL_ARM}, held-out ---")
        print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())